# Lab03D — Inferência complementar e previsão histórica

O laboratório enfrenta desvios de inferência estática, amplia a calibração temporal e acrescenta previsão do **retorno logarítmico do ativo na próxima sessão**, com informação estritamente anterior ao alvo. Não usa preços como variável de regressão. Amostras fixas80/117, dez setores, classes preservadas.

Execute **Ambiente de execução → Executar tudo**, CPU. Reutiliza dois ZIPs autenticados do projeto, sem coletar cotações ou repetir os labs anteriores. As simulações cobrem todos os casos em cada réplica. Há progresso por lote; geração do banco e conferências podem levar dezenas de minutos conforme a CPU.

**Estado:** preparação local calculada e verificada; execução Colab deste notebook ainda pendente. As escolhas financeiras já eram conhecidas; planos e adendos explicitam a sequência de decisões, sem pré-registro independente. Não se promete eliminar toda limitação ou obter vantagem preditiva.

In [ ]:
import os,sys,json,hashlib,zipfile,subprocess,shutil,importlib.util
from pathlib import Path
from datetime import datetime,timezone
for module,package in [('fitz','pymupdf'),('PIL','pillow'),('threadpoolctl','threadpoolctl')]:
    if importlib.util.find_spec(module) is None:subprocess.run([sys.executable,'-m','pip','install','--quiet',package],check=True)
import numpy as np,pandas as pd
try:
    from IPython.display import display,Image
except ImportError:
    def display(value):print(str(value)[:5000])
    def Image(filename):
        from PIL import Image as PillowImage
        image=PillowImage.open(filename);image.load();return filename
LOCAL_MODE=bool(os.environ.get('ALEM_BETA_LOCAL'))
os.environ['ALEM_BETA_EXECUTION_ENV']='local cached notebook verification' if LOCAL_MODE else 'Colab'
BASE=Path(os.environ.get('ALEM_BETA_LOCAL','/content/alem_beta_lab03d')).resolve();BASE.mkdir(parents=True,exist_ok=True)
RUN=BASE/('run_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'));RUN.mkdir();ROOT=RUN/'runtime'
print('Lab03D |',os.environ['ALEM_BETA_EXECUTION_ENV'],'| CPU |',RUN.name)

## Caches e proveniência

ZIPs X e B são autenticados integralmente e por manifesto interno. A referência conserva sete NaNs na década, três nos80comuns recentes e sete nos117recentes. Incluem os cinco retornos de direitos não identificáveis e lacunas herdadas ANIM3/AZUL4. Sem interpolação ou pontes distribuídas. Light reconciliada; exclusão2024Q4 é apenas sensibilidade.

In [ ]:
RUNTIME_ID = '1u5_My2iJgdcnx8VDFnpqkaYd4YXlA7gD'
RUNTIME_SHA = '2f897764224f1e225e7bcd778eac2ee2127953d3946a09fff08998af9ed05879'
X_ID = '1aExT7v-GD8s5xKzM0K-0LIIJER8E8Twg'
B_ID = '1hhLvxQpAnEGopDvnt1Mh9IbkV0gYt9Gb'
X_SHA = '297da2afde534082c398324f308df7af134c9692e7d61d6e0b2b820341ea5214'
B_SHA = '645946c0bfee05cc7cbbc34b63af74700fcc7e0a3d3ac14ee7aaa398323fa26d'
FOLDER_ID = '1L-cV8Zp0UPKPDkICoMoBk_LZuWSjzK53'
def digest(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
if not LOCAL_MODE:
    from google.colab import auth, drive
    auth.authenticate_user()
    drive.mount('/content/drive')
    import google.auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
    credentials, _ = google.auth.default()
    service = build('drive', 'v3', credentials=credentials, cache_discovery=False)
    CACHE = Path('/content/drive/MyDrive/alem_do_beta/lab03d_cache')
    CACHE.mkdir(parents=True, exist_ok=True)
else:
    CACHE = BASE / 'cache'; CACHE.mkdir(exist_ok=True)
    LOCAL_FILES = {RUNTIME_ID:BASE/'lab03d_runtime_v1.zip', X_ID:BASE.parent/'work_lab03b_colab/x.zip', B_ID:BASE.parent/'work_lab03b_colab/results.zip'}
def obtain(file_id, expected):
    if LOCAL_MODE:
        p = LOCAL_FILES[file_id]; assert digest(p)==expected; return p
    cached = CACHE/(file_id+'.zip')
    if cached.exists() and digest(cached)==expected:
        print('Cache autenticado:',file_id); return cached
    name = service.files().get(fileId=file_id,fields='name').execute()['name']
    # Existing exports anywhere in MyDrive are tried by exact name and hash.
    for candidate in Path('/content/drive/MyDrive').rglob(name):
        if candidate.is_file() and digest(candidate)==expected:
            print('Export existente autenticado:',name); return candidate
    stage = cached.with_suffix('.part')
    with stage.open('wb') as stream:
        request = service.files().get_media(fileId=file_id)
        download = MediaIoBaseDownload(stream,request); done=False
        while not done: _,done=download.next_chunk()
    assert digest(stage)==expected, 'SHA256 divergente; não utilizar arquivo'
    stage.replace(cached); print('Artefato de projeto autenticado:',name); return cached
RUNTIME = obtain(RUNTIME_ID,RUNTIME_SHA)
X_ARCHIVE = obtain(X_ID,X_SHA)
B_ARCHIVE = obtain(B_ID,B_SHA)

In [ ]:
ROOT.mkdir()
with zipfile.ZipFile(RUNTIME) as z:
    names=z.namelist();assert len(names)==len(set(names))
    for n in names:assert not n.startswith('/') and '..' not in Path(n).parts
    z.extractall(ROOT)
source_manifest=json.loads((ROOT/'data/lab03d/runtime_manifest.json').read_text())
for name,expected in source_manifest.items():assert digest(ROOT/name)==expected,name
assert set(names)==set(source_manifest)|{'data/lab03d/runtime_manifest.json'}
lock=json.loads((ROOT/'data/lab03d/inputs_manifest.json').read_text());assert digest(ROOT/'data/lab03d/plano_v1.json')==lock['plan_sha256']
sys.path.insert(0,str(ROOT/'src'))
def script(name,*args):subprocess.run([sys.executable,'-u',str(ROOT/'scripts'/name),*map(str,args)],check=True)
print(len(source_manifest),'fontes autenticadas; protocolos e adendos preservados.')

## Desenho e controles temporais

Primário histórico: HAC5/Holm197, famílias temporais474 e estimativas pontuais preservados. Complementos: HC3-HAC5/Holm, maxT gaussiano999, envelope de postos MC máximo sobre três mecanismos. Cada banco tem9999 simulações por mecanismo; validação conjunta independente1000. Desenhos equivalentes reduzem apenas simulação redundante de banco: sete estáticos,14anuais, quatroθ; validação mantém117 caminhos e dependência.

Previsão: zero, média, tradicional e condicional. Janelas252primária/504sensibilidade, atualização mensal, mínimo200pares/60por regime/95%/variância; insuficiência recua ambos os modelos de mercado à média, mantendo o ativo. Mercado AR1 e inovações são ajustados no passado; integra-se a regressão condicional sobre suas inovações. **Não se conhece mercado/sinal do dia previsto.** Mercado anterior é atualizado diariamente. 80ativos:2019–2025;117:2024–2025;80comuns reutilizados em2024–2025. O comum descritivo continua2021–2025.

Oito configurações incluem eventos, caixa, Light e proteção causal de lacunas. Esta última só conhece ausências até cada origem; nenhuma lacuna futura pode modificar treino ou filtro anterior. São cenários condicionais, sem homologação global de ajustes.

In [ ]:
check=subprocess.run([sys.executable,'-m','unittest','discover','-s',str(ROOT/'tests'),'-p','test_lab03d.py','-v'],capture_output=True,text=True)
print(check.stdout+check.stderr);assert check.returncode==0
from lab03d_inference import load
prepared=load(ROOT,X_ARCHIVE,B_ARCHIVE)
display(prepared['design'].groupby('scenario').agg(ativos=('papel_id','size'),ausencias=('n_missing','sum')))
assert prepared['design'].groupby('scenario').n_missing.sum().to_dict()=={'decada':7,'recente':7}

## Simulação integral e previsão

Três mecanismos: normal; t5 com escala por mercado defasado; mesma t5 com AR0,3. Preservam-se os calendários e as máscaras. O envelope protege somente essa classe finita, sem garantia livre de distribuição ou identificação do mecanismo real. O banco é independente das réplicas de validação. Nenhum resultado desfavorável é descartado.

In [ ]:
OUTPUT=RUN/'outputs'
# This local-only route verifies the notebook against the already computed full run.
# Colab always executes both generators from scratch; this route cannot activate there.
if LOCAL_MODE and os.environ.get('ALEM_BETA_LOCAL_CACHE'):
    shutil.copytree(Path(os.environ['ALEM_BETA_LOCAL_CACHE']),OUTPUT)
    print('Conferência local de resultados já calculados; não é nova execução integral nem Colab.')
else:
    script('run_lab03d.py','--x-archive',X_ARCHIVE,'--b-archive',B_ARCHIVE,'--output',OUTPUT)
print((OUTPUT/'metricas.json').read_text())

In [ ]:
if not (LOCAL_MODE and os.environ.get('ALEM_BETA_LOCAL_CACHE')):
    script('run_lab03d_rank.py','--x-archive',X_ARCHIVE,'--b-archive',B_ARCHIVE,'--output',OUTPUT)
print((OUTPUT/'metricas_envelope.json').read_text())
script('run_lab03d_lfc.py','--output',OUTPUT)

## Resultados visuais e precisão

As figuras mostram cobertura/FWER, todos os anos preditivos, sensibilidades e intervalos por ativo. Barras de simulação são incerteza Monte Carlo, não incerteza dos retornos reais. Intervalos preditivos de ganho de MSE são **marginais exploratórios**, em blocos21/42/63, condicionais às perdas registradas; não se validou sua cobertura nominal nem se afirma superioridade após multiplicidade.

O piso de valor-p de postos é0,0001: mínimos ajustados são0,0197 na família197 e0,0474 na474. Exportam-se contagens e intervalos Wilson pontuais da probabilidade de cauda. Não são intervalos do valor-p discreto, simultâneos ou autorização para aumentar o banco até rejeitar.

In [ ]:
script('plot_lab03d.py','--output',OUTPUT)
for name in ['lab03d_calibracao','lab03d_previsao_anual','lab03d_sensibilidades_previsao','lab03d_intervalos_ativos','lab03d_iut_limite']:
    display(Image(filename=str(OUTPUT/'figures'/(name+'.png'))))

In [ ]:
family=pd.read_csv(OUTPUT/'envelope_family.csv');display(family[family.pattern.eq('dense_+0.00')][['mechanism','fwer_adjusted','fwer_adjusted_mc_low','fwer_adjusted_mc_high']])
gains=pd.read_csv(OUTPUT/'prediction_gains.csv');q=gains[gains.configuration.eq('N252')&gains.period.eq('full')&gains.regime.eq('all')]
display(q.groupby('scenario').relative_mse_gain.agg(['count','median','min','max']))
display(pd.read_csv(OUTPUT/'observed_envelope.csv').query('p_envelope_holm_197 < .05')[['ticker','scenario','delta','p_envelope_holm_197']])

## Conferência independente

Três verificadores separados da implementação de produção: regressões completas e sandwiches, previsões e métricas, sorteios por índices, banco de postos, famílias e diagnósticos IUT. Os testes perturbam alvos e informações futuras para garantir invariância de previsões anteriores. Aprovação numérica não significa homologação econômica ou validade inferencial irrestrita.

In [ ]:
script('validate_lab03d_outputs.py','--output',OUTPUT,'--x-archive',X_ARCHIVE,'--b-archive',B_ARCHIVE)
script('validate_lab03d_rank.py','--output',OUTPUT,'--x-archive',X_ARCHIVE)
script('validate_lab03d_diagnostics.py','--output',OUTPUT)
for name in ['validacao_independente.json','validacao_envelope_independente.json','validacao_diagnosticos_independente.json']:
    record=json.loads((OUTPUT/name).read_text());assert record['status']=='passed';print(name,json.dumps(record,indent=2))

## Alcance e exportação

A preparação local encontrou melhoria de calibração sob a classe declarada, mas HC3/maxT sozinhos não resolveram todos os cenários. Nos retornos, o envelope conserva somente VULC3 na década; decisões originais HAC5 continuam identificadas como aproximação. Não rejeição temporal não prova equivalência. Ganhos preditivos medianos são pequenos e mistos, sem vantagem generalizada ou estratégia lucrativa demonstrada.

Continuam: composição retrospectiva2025/sobrevivência; vintages do fornecedor não reconstruídos; auditoria econômica global0/197, taxas0/65 e líquidos0/48; lacunas não identificáveis; setores unitários; ausência de holdout prospectivo intocado e de calibração dos ICs preditivos. O protocolo foi ampliado após resultados iniciais conhecidos, com adendos e sementes novas documentados.

O export completo inclui réplicas, previsões, tabelas, cinco figurasPNG/PDF, fontes por hash, protocolos e versões; pode superar150MB. **Salve este notebook após executar tudo**, além do ZIP que será salvo automaticamente na pasta Lab03D.

Referências metodológicas: MacKinnon e White1985, DOI10.1016/0304-4076(85)90158-7 (metadados/resumo); Shao2010, DOI10.1198/jasa.2009.tm08744; Chernozhukov et al.2013, DOI10.1214/13-AOS1161; Dufour2006, DOI10.1016/j.jeconom.2005.06.007; Phipson e Smyth2010, DOI10.2202/1544-6115.1585; Hyndman e Athanasopoulos2021, https://otexts.com/fpp3/tscv.html. Nenhum desses trabalhos certifica automaticamente o híbrido proposto sob qualquer DGP financeiro.

In [ ]:
STAMP=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ');EXPORT=RUN/('resultados_lab03d_'+STAMP+'.zip')
script('export_lab03d.py','--output',OUTPUT,'--destination',EXPORT)
if not LOCAL_MODE:
    saved=service.files().create(body={'name':EXPORT.name,'parents':[FOLDER_ID]},media_body=MediaFileUpload(str(EXPORT),mimetype='application/zip',resumable=True),fields='id,name,webViewLink').execute()
    print('Resultado salvo no Drive:',saved.get('webViewLink','https://drive.google.com/file/d/'+saved['id']+'/view'))
else:print('Conferência local; Colab ainda pendente.')
print('SHA256:',digest(EXPORT));print('Concluído. Salve o notebook no Colab.')